# 04 — Parallel Tempering (Replica Exchange NUTS)

**Hypothesis H3**: Parallel tempering with a geometric temperature ladder
overcomes the singular geometry that defeats fixed mass-matrix NUTS.

**Mechanism**: Hot chains (β < 1) flatten the posterior, making degenerate
directions easier to traverse. Replica-exchange swaps propagate this
exploration to the cold chain (β = 1) which targets the true posterior.

**Prediction**: The cold chain's ACF@50 median will drop below 0.3 (vs ~0.97
in H1), and ESS will increase by 10–100× compared to single-chain NUTS.

**Temperature ladder**: β = [1.0, 0.7, 0.5, 0.3, 0.1] — geometric spacing
with 5 rungs. Swap proposals every 10 samples (DEO scheme).

**Success criterion**:
- ACF@50 median < 0.3 → H3 confirmed (tempering breaks the mixing barrier)
- ACF@50 median > 0.5 → H3 falsified (even tempering can't fix it)

In [ ]:
import collections
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root
if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
RESULTS_DIR = DATA_DIR / "results_tempered"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
# Load dataset
data = torch.load(DATA_DIR / "sequences.pt", weights_only=False)
sequences = data["sequences"]
data_cfg = data["config"]

VOCAB_SIZE = int(data_cfg["n_states"])
MAX_LEN = int(data_cfg["L"])
PAD_ID = int(data_cfg.get("pad_id", -1))
DGP_REGIME = data_cfg.get("dgp_regime", "unknown")

print(f"DGP regime: {DGP_REGIME}")
print(
    f"Sequences: {tuple(sequences.shape)}, VOCAB_SIZE={VOCAB_SIZE}, MAX_LEN={MAX_LEN}"
)

# Load trained model from checkpoint
device = torch.device("cpu")
D_MODEL = VOCAB_SIZE * 2

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
).to(device)

ckpt = torch.load(DATA_DIR / "checkpoint_single_chain.pt", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
print(f"Total parameters: {total_params:,}")

# Prepare data tensors
x_data = sequences[:, :-1].to(device)
y_data = sequences[:, 1:].to(device)
x_data = x_data.clone()
x_data[x_data == PAD_ID] = 0

mle_param = torch.cat([p.flatten() for p in model.parameters()]).detach()
print(f"MLE parameter vector: d = {mle_param.shape[0]}")

## Define Log-Likelihood and Prior

In [ ]:
from torch_bdn.bn import BayesianNet
from torch_bdn.sampling import NUTS, Perturb, Sampler

SIGMA_PRIOR = 10.0


def loss_fn(logits, targets):
    ce = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
        reduction="none",
        ignore_index=PAD_ID,
    )
    ce = ce.view(logits.shape[:-1])
    mask = (targets != PAD_ID).float()
    return (ce * mask).sum() / mask.sum()


def make_prior_logp(mu: torch.Tensor, sigma=10.0):
    """Gaussian prior N(mu, sigma^2 I) — centred at the MAP."""
    mean = mu.detach().clone()

    def prior_logp(params):
        flat = torch.cat([p.flatten() for p in params])
        diff = flat - mean
        return -0.5 * diff.pow(2).sum() / (sigma**2)

    return prior_logp


bn = BayesianNet(
    model, loss_fn, make_prior_logp(mle_param, sigma=SIGMA_PRIOR), compile=True
)
print(f"BayesianNet ready: d={mle_param.shape[0]}, prior σ={SIGMA_PRIOR}")

## Parallel Tempering Schedule

**Temperature ladder design**: Geometric spacing with 5 rungs.
- β = 1.0 — cold chain (true posterior)
- β = 0.7 — mild tempering
- β = 0.5 — moderate tempering
- β = 0.3 — strong tempering
- β = 0.1 — hot chain (nearly flat posterior)

**Physics**: At inverse temperature β, the posterior becomes
π_β(θ) ∝ p(D|θ)^β · p(θ). The log-density curvature scales as β·H,
so degenerate directions (H≈0) remain flat while non-degenerate
directions become shallower. The hot chains can cross ridges and
explore the full manifold.

**Swap frequency**: Every 10 NUTS samples, propose DEO adjacent swaps.
This balances swap frequency against per-chain decorrelation.

In [ ]:
# ── Tempering configuration ──
BETAS = [1.0, 0.7, 0.5, 0.3, 0.1]  # inverse temperatures (geometric-ish ladder)
N_CHAINS = len(BETAS)
SWAP_EVERY = 10  # propose swaps every K samples
N_WARMUP_PT = 5000  # warmup per chain (each adapts its own M independently)
N_SAMPLES_PT = 5000  # production samples per chain
MAX_TREE_DEPTH = 7
TARGET_ACCEPT = 0.69

print(f"Temperature ladder: β = {BETAS}")
print(f"  {N_CHAINS} chains, swap every {SWAP_EVERY} samples")
print(f"  Warmup: {N_WARMUP_PT}, Production: {N_SAMPLES_PT}")
print(f"  Tree depth: {MAX_TREE_DEPTH}, target accept: {TARGET_ACCEPT}")
print("\n  Effective curvature scaling at each β:")
for beta in BETAS:
    print(f"    β={beta:.1f} → H_eff = {beta:.1f}·H  (ridge height × {beta:.1f})")

## Configure and Run Tempered NUTS

Using `torch_bdn`'s built-in parallel tempering: `sampler.sample(...)` with
`betas=[...]` and `swap_every=K`. The API handles:
- Independent NUTS per chain at each temperature
- DEO (Deterministic Even-Odd) swap proposals between adjacent rungs
- Replica-exchange Metropolis-Hastings acceptance criterion
- Tracking swap acceptance counts

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PARALLEL TEMPERING SAMPLING
# ══════════════════════════════════════════════════════════════════════════════
import time

sampler = Sampler(bn, x_data, y_data)

print(f"Starting parallel tempering: {N_CHAINS} chains × {N_SAMPLES_PT} samples")
print(f"  β = {BETAS}, swap_every = {SWAP_EVERY}")
print(f"  This will take a while (d={mle_param.shape[0]}, depth={MAX_TREE_DEPTH})...")

t0 = time.time()

result = sampler.sample(
    config=NUTS(
        n_warmup=N_WARMUP_PT,
        step_size=0.01,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=TARGET_ACCEPT,
        adapt_mass_matrix=True,
    ),
    n_samples=N_SAMPLES_PT,
    n_chains=N_CHAINS,
    init_strategy=Perturb(scale=1.0),
    swap_every=SWAP_EVERY,
    betas=BETAS,
)

elapsed = time.time() - t0
print(f"\n✓ Sampling complete in {elapsed / 60:.1f} min")

# ── Swap evidence from swap_history ──
# The sampler records every swap proposal in result.swap_history:
#   {"round": int, "pair": (i, j), "accepted": bool, "log_alpha": float}
print("\n  Swap log:")
print(f"    Total proposals: {result.n_swaps_proposed}")
print(f"    Accepted: {result.n_swaps_accepted}")
print(f"    Acceptance rate: {result.swap_acceptance_rate():.3f}")

# Per-pair breakdown from swap_history
if result.swap_history:
    from collections import Counter

    pair_proposed = Counter()
    pair_accepted = Counter()
    for entry in result.swap_history:
        pair = entry["pair"]
        pair_proposed[pair] += 1
        if entry["accepted"]:
            pair_accepted[pair] += 1

    print("\n    Per-pair swap rates:")
    for pair in sorted(pair_proposed.keys()):
        i, j = pair
        n_prop = pair_proposed[pair]
        n_acc = pair_accepted[pair]
        rate = n_acc / n_prop if n_prop > 0 else 0
        print(
            f"      β={BETAS[i]:.1f} ↔ β={BETAS[j]:.1f}: {n_acc}/{n_prop} = {rate:.3f}"
        )

    # Show first few swap events as evidence
    accepted_swaps = [e for e in result.swap_history if e["accepted"]]
    print(f"\n    First 10 accepted swaps (of {len(accepted_swaps)} total):")
    for e in accepted_swaps[:10]:
        i, j = e["pair"]
        print(
            f"      round {e['round']:>4d}: β={BETAS[i]:.1f} ↔ β={BETAS[j]:.1f}  (log α = {e['log_alpha']:.2f})"
        )
else:
    print("    ⚠ No swap_history available")

# Per-chain summary (β read from diagnostics)
print("\n  Per-chain summary:")
for ci, ch in enumerate(result.chains):
    diag = ch.diagnostics
    beta_reported = diag.get("beta", None)
    beta_str = (
        f"β={beta_reported:.2f}" if beta_reported is not None else "β=? (not in diag)"
    )
    eps = diag.get("adapted_step_size", diag.get("step_size", "?"))
    print(
        f"    Chain {ci} ({beta_str}): "
        f"accept={ch.acceptance_rate:.3f}, "
        f"ε={eps:.4e}, "
        f"mean_depth={diag.get('mean_tree_depth', 0):.1f}"
    )

In [ ]:
# ── Persist tempered samples ──
tempered_path = DATA_DIR / "nuts_samples_tempered.pt"

chains_payload = [
    {
        "parameters": torch.stack(ch.parameters).cpu(),
        "acceptance_rate": ch.acceptance_rate,
        "diagnostics": ch.diagnostics,
        "beta": BETAS[ci],
    }
    for ci, ch in enumerate(result.chains)
]

torch.save(
    {
        "chains": chains_payload,
        "config": {
            "n_chains": N_CHAINS,
            "betas": BETAS,
            "swap_every": SWAP_EVERY,
            "n_warmup": N_WARMUP_PT,
            "n_samples": N_SAMPLES_PT,
            "max_tree_depth": MAX_TREE_DEPTH,
            "sigma_prior": SIGMA_PRIOR,
            "dgp_regime": DGP_REGIME,
            "target_accept": TARGET_ACCEPT,
        },
        "swap_diagnostics": {
            "n_swaps_proposed": result.n_swaps_proposed,
            "n_swaps_accepted": result.n_swaps_accepted,
            "swap_acceptance_rate": result.swap_acceptance_rate(),
        },
        "mle_param": mle_param.cpu(),
        "elapsed_seconds": elapsed,
    },
    tempered_path,
)
print(f"✓ Saved to {tempered_path}")
print(f"  {N_CHAINS} chains × {N_SAMPLES_PT} samples × d={mle_param.shape[0]}")

## Trace Diagnostics: Swap Acceptance and Round-Trip Rates

Key diagnostics for tempering health:
1. **Swap acceptance rate** — should be 20–60% between adjacent rungs
2. **Round-trip rate** — how often a configuration travels cold→hot→cold
3. If acceptance < 20% between any pair, ladder spacing is too coarse

In [ ]:
# ── Swap diagnostics ──
swap_rate = result.swap_acceptance_rate()
print(f"Overall swap acceptance rate: {swap_rate:.3f}")
print(f"  Proposed: {result.n_swaps_proposed}, Accepted: {result.n_swaps_accepted}")

# Per-rung pair acceptance (if available in diagnostics)
# The DEO scheme alternates even/odd pairs: (0,1),(2,3),... then (1,2),(3,4),...
n_pairs = N_CHAINS - 1
print(f"\nTemperature pairs ({n_pairs} adjacent):")
for i in range(n_pairs):
    print(
        f"  β={BETAS[i]:.1f} ↔ β={BETAS[i + 1]:.1f}  (ΔE scaling ~ {BETAS[i] - BETAS[i + 1]:.2f})"
    )

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
bar_labels = [f"{BETAS[i]:.1f}↔{BETAS[i + 1]:.1f}" for i in range(n_pairs)]

# If per-pair rates are available, use them; otherwise show global rate
ax.bar(bar_labels, [swap_rate] * n_pairs, color="steelblue", alpha=0.8)
ax.axhline(0.2, color="red", ls="--", lw=1.5, label="Minimum viable (20%)")
ax.axhline(0.5, color="green", ls=":", lw=1.5, label="Ideal target (50%)")
ax.set_xlabel("Temperature pair", fontsize=12)
ax.set_ylabel("Swap acceptance rate", fontsize=12)
ax.set_title("Replica Exchange Swap Acceptance", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Health check
if swap_rate < 0.1:
    print(
        "\n⚠ WARNING: Swap rate < 10% — ladder too coarse, need intermediate temperatures"
    )
elif swap_rate < 0.2:
    print("\n⚠ CAUTION: Swap rate < 20% — tempering may be inefficient")
else:
    print(f"\n✓ Swap rate {swap_rate:.1%} — healthy tempering")

## ACF Analysis in Hessian Eigenbasis

Compute the Hessian eigenbasis (for diagnostics only — not used in sampling).
Project the **cold chain** (β=1.0) samples into this basis and compute the
autocorrelation function per direction.

Compare degenerate vs non-degenerate directions: if tempering works, the
degenerate directions should show dramatically faster ACF decay than H1.

In [ ]:
# Compute Hessian eigenbasis FOR DIAGNOSTICS ONLY
from torch_bdn.bn.bayesian_net import approx_hessian

print("Computing Hessian eigenbasis for ACF diagnostic...")
H_mle = approx_hessian(model, loss_fn, x_data, y_data, chunk_size=10240).cpu()
eigvals, eigvecs = torch.linalg.eigh(H_mle)

eigval_threshold = eigvals.max().item() * 1e-3
n_nd = int((eigvals.abs() > eigval_threshold).sum())
n_degen = eigvals.shape[0] - n_nd
print(f"Hessian: λ ∈ [{eigvals[0]:.2e}, {eigvals[-1]:.2e}]")
print(f"  {n_nd} non-degenerate directions, {n_degen} degenerate directions")

In [ ]:
# ── ACF per Hessian eigendirection (cold chain only) ──
V = eigvecs.cpu().float()
MAX_LAG = min(2000, N_SAMPLES_PT // 4)

sort_idx = torch.argsort(eigvals.cpu(), descending=True)
eigvals_sorted = eigvals.cpu()[sort_idx]


def compute_acf_eigenbasis(samples_tensor, V, max_lag):
    """Project samples into eigenbasis, compute ACF per direction via FFT."""
    z = samples_tensor @ V
    d = z.shape[1]
    acf = torch.zeros(d, max_lag)
    for j in range(d):
        x = z[:, j]
        x = x - x.mean()
        var = x.var()
        if var < 1e-20:
            continue
        n = x.shape[0]
        padded = torch.zeros(2 * n)
        padded[:n] = x
        ft = torch.fft.rfft(padded)
        acov = torch.fft.irfft(ft * ft.conj())[:n] / n
        acf[j, :max_lag] = acov[:max_lag] / acov[0].clamp(min=1e-20)
    return acf


# Cold chain = chain 0 (β = 1.0)
cold_samples = torch.stack(result.chains[0].parameters).cpu().float()
acf_sorted = compute_acf_eigenbasis(cold_samples, V, MAX_LAG)[sort_idx]

# ── ACF heatmap ──
fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(
    acf_sorted.numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.3,
    vmax=1.0,
    interpolation="nearest",
    origin="upper",
)
ax.set_xlabel("Lag τ", fontsize=12)
ax.set_ylabel("Eigendirection (sorted by λ, largest at top)", fontsize=12)
ax.set_title(
    f"ACF per Hessian eigendirection — Tempered NUTS (cold chain β=1.0)\n"
    f"warmup={N_WARMUP_PT}, {N_SAMPLES_PT} samples, d={mle_param.shape[0]}",
    fontsize=13,
    fontweight="bold",
)

if 0 < n_nd < cold_samples.shape[1]:
    ax.axhline(
        n_nd - 0.5,
        color="lime",
        lw=2,
        ls="--",
        label=f"degen boundary (top {n_nd} non-degen)",
    )
    ax.legend(loc="upper right", fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8, label="ACF")
plt.tight_layout()
plt.show()

# Summary statistics
acf_at_10 = acf_sorted[:, min(10, MAX_LAG - 1)]
acf_at_50 = acf_sorted[:, min(50, MAX_LAG - 1)]
acf_at_200 = acf_sorted[:, min(200, MAX_LAG - 1)]
print(f"\nACF at lag 10:   median={acf_at_10.median():.3f},  max={acf_at_10.max():.3f}")
print(f"ACF at lag 50:   median={acf_at_50.median():.3f},  max={acf_at_50.max():.3f}")
print(f"ACF at lag 200:  median={acf_at_200.median():.3f},  max={acf_at_200.max():.3f}")

# Separate stats for non-degen vs degen
acf50_nd = acf_sorted[:n_nd, min(50, MAX_LAG - 1)]
acf50_dg = acf_sorted[n_nd:, min(50, MAX_LAG - 1)]
print(
    f"\nACF@50 non-degen ({n_nd} dirs): median={acf50_nd.median():.3f}, max={acf50_nd.max():.3f}"
)
print(
    f"ACF@50 degenerate ({n_degen} dirs): median={acf50_dg.median():.3f}, max={acf50_dg.max():.3f}"
)

## ESS Comparison: Tempered vs Untempered Baseline

Load H1 baseline ESS from `results_no_hessian/` and compare against the
tempered cold chain. The improvement factor quantifies how much tempering helps.

In [ ]:
# ── ESS estimate (cold chain) ──
def compute_ess(acf_row):
    """Estimate ESS from a single ACF row using initial positive sequence."""
    total = 0.0
    for k in range(acf_row.shape[0]):
        if acf_row[k] < 0:
            break
        total += acf_row[k]
    tau = 1 + 2 * (total - 1)
    return N_SAMPLES_PT / max(tau, 1.0)


ess_tempered = torch.tensor(
    [compute_ess(acf_sorted[i]) for i in range(acf_sorted.shape[0])]
)
ess_nd_t = ess_tempered[:n_nd]
ess_dg_t = ess_tempered[n_nd:]

print(f"ESS (tempered cold chain, N={N_SAMPLES_PT}):")
print(
    f"  All:       min={ess_tempered.min():.1f}, median={ess_tempered.median():.1f}, max={ess_tempered.max():.1f}"
)
print(
    f"  Non-degen: min={ess_nd_t.min():.1f}, median={ess_nd_t.median():.1f}, max={ess_nd_t.max():.1f}"
)
print(
    f"  Degenerate: min={ess_dg_t.min():.1f}, median={ess_dg_t.median():.1f}, max={ess_dg_t.max():.1f}"
)

# ── Load H1 baseline ──
baseline_dir = DATA_DIR / "results_no_hessian"
baseline_tensors = torch.load(
    baseline_dir / "diagnostics_tensors.pt", weights_only=True
)
ess_baseline = baseline_tensors["ess_per_dir"]

print("\nESS (H1 baseline, N=10000):")
print(
    f"  All:       min={ess_baseline.min():.1f}, median={ess_baseline.median():.1f}, max={ess_baseline.max():.1f}"
)

# ── Improvement factor ──
# Normalize to per-sample ESS rate for fair comparison
ess_rate_tempered = ess_tempered / N_SAMPLES_PT
ess_rate_baseline = ess_baseline / 10000  # H1 used 10k samples

improvement = ess_rate_tempered / ess_rate_baseline.clamp(min=1e-6)
print("\nESS rate improvement (tempered / H1):")
print(
    f"  min={improvement.min():.2f}×, median={improvement.median():.2f}×, max={improvement.max():.2f}×"
)

# ── Side-by-side bar plot ──
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].bar(
    range(len(ess_tempered)),
    ess_tempered.numpy(),
    width=1.0,
    color="darkorange",
    alpha=0.7,
    label=f"Tempered (N={N_SAMPLES_PT})",
)
if 0 < n_nd < len(ess_tempered):
    axes[0].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes[0].set_ylabel("ESS")
axes[0].set_title("H3: Tempered NUTS (cold chain)", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].set_xlim(-0.5, len(ess_tempered) - 0.5)
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(
    range(len(ess_baseline)),
    ess_baseline.numpy(),
    width=1.0,
    color="steelblue",
    alpha=0.7,
    label="H1 baseline (N=10000)",
)
if 0 < n_nd < len(ess_baseline):
    axes[1].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes[1].set_xlabel("Eigendirection (sorted by λ, largest first)")
axes[1].set_ylabel("ESS")
axes[1].set_title("H1: Single-chain NUTS (baseline)", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].set_xlim(-0.5, len(ess_baseline) - 0.5)
axes[1].grid(True, alpha=0.3, axis="y")

fig.suptitle("ESS Comparison: Tempered vs Untempered", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Spectral Diagnostics at Cold Chain

Compute the sample covariance of cold-chain draws and compare its
eigenspectrum to the Hessian. If tempering is effective, the sample
covariance should show variance in degenerate directions (which were
unexplored in H1).

In [ ]:
# ── Spectral analysis of cold-chain sample covariance ──
# Project samples into Hessian eigenbasis
z_cold = cold_samples @ V  # (N, d) in eigenbasis
z_cold_centered = z_cold - z_cold.mean(dim=0, keepdim=True)

# Per-direction variance in eigenbasis
var_per_dir = z_cold_centered.var(dim=0)  # shape (d,)
var_sorted = var_per_dir[sort_idx]

# Expected variance from posterior:
# Non-degen: σ² ≈ 1/(n·λ_H + 1/σ_prior²) ≈ 1/(n·λ_H) for large data
# Degen: σ² ≈ σ_prior² = 100 (prior dominates)
prior_var = SIGMA_PRIOR**2
expected_var_degen = prior_var  # should be ~100 if fully explored

# Hessian eigenvalues (absolute, sorted descending)
eig_h_sorted = eigvals_sorted.abs()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: sample variance per eigendirection
rank_idx = torch.arange(1, len(var_sorted) + 1)
ax1.semilogy(
    rank_idx.numpy(),
    var_sorted.numpy(),
    "o",
    ms=2,
    color="teal",
    alpha=0.6,
    label="Sample variance (tempered cold chain)",
)
ax1.axhline(
    prior_var, color="red", ls="--", lw=1.5, label=f"Prior variance = {prior_var}"
)
if 0 < n_nd < len(var_sorted):
    ax1.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7, label="degen boundary")
ax1.set_xlabel("Eigendirection rank", fontsize=11)
ax1.set_ylabel("Variance (log)", fontsize=11)
ax1.set_title(
    "Sample variance per Hessian eigendirection", fontsize=12, fontweight="bold"
)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: compare tempered vs H1 variance
# Load H1 samples for comparison
h1_data = torch.load(DATA_DIR / "nuts_samples_no_hessian.pt", weights_only=False)
h1_samples = h1_data["chains"][0]["parameters"].float()
z_h1 = h1_samples @ V
var_h1 = (z_h1 - z_h1.mean(dim=0, keepdim=True)).var(dim=0)[sort_idx]

ax2.semilogy(
    rank_idx.numpy(),
    var_sorted.numpy(),
    "o",
    ms=2,
    color="darkorange",
    alpha=0.6,
    label="Tempered cold chain",
)
ax2.semilogy(
    rank_idx.numpy(),
    var_h1.numpy(),
    "o",
    ms=2,
    color="steelblue",
    alpha=0.4,
    label="H1 baseline",
)
ax2.axhline(prior_var, color="red", ls="--", lw=1, alpha=0.5)
if 0 < n_nd < len(var_sorted):
    ax2.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7)
ax2.set_xlabel("Eigendirection rank", fontsize=11)
ax2.set_ylabel("Variance (log)", fontsize=11)
ax2.set_title("Variance comparison: Tempered vs H1", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

fig.suptitle(
    "Spectral diagnostics: cold-chain exploration", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Summary
var_ratio = var_sorted / var_h1.clamp(min=1e-10)
print("Variance ratio (tempered / H1):")
print(f"  Non-degen: median = {var_ratio[:n_nd].median():.2f}×")
print(f"  Degenerate: median = {var_ratio[n_nd:].median():.2f}×")
print(
    f"  Degen directions at prior variance: {(var_sorted[n_nd:] > prior_var * 0.5).sum()}/{n_degen}"
)

## Save Results and Verdict

Save all figures and diagnostics. Compare against H1 baseline to determine
whether parallel tempering breaks the mixing barrier.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE ALL RESULTS
# ══════════════════════════════════════════════════════════════════════════════

# --- Re-create and save ACF heatmap ---
fig_acf, ax_acf = plt.subplots(figsize=(16, 8))
im_acf = ax_acf.imshow(
    acf_sorted.numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-0.3,
    vmax=1.0,
    interpolation="nearest",
    origin="upper",
)
ax_acf.set_xlabel("Lag τ", fontsize=12)
ax_acf.set_ylabel("Eigendirection (sorted by λ, largest at top)", fontsize=12)
ax_acf.set_title(
    f"ACF per Hessian eigendirection — Tempered NUTS (cold chain β=1.0)\n"
    f"warmup={N_WARMUP_PT}, {N_SAMPLES_PT} samples, d={mle_param.shape[0]}",
    fontsize=13,
    fontweight="bold",
)
if 0 < n_nd < acf_sorted.shape[0]:
    ax_acf.axhline(
        n_nd - 0.5,
        color="lime",
        lw=2,
        ls="--",
        label=f"degen boundary (top {n_nd} non-degen)",
    )
    ax_acf.legend(loc="upper right", fontsize=10)
plt.colorbar(im_acf, ax=ax_acf, shrink=0.8, label="ACF")
fig_acf.tight_layout()
fig_acf.savefig(RESULTS_DIR / "acf_heatmap_tempered.pdf", bbox_inches="tight", dpi=150)
fig_acf.savefig(RESULTS_DIR / "acf_heatmap_tempered.png", bbox_inches="tight", dpi=150)
plt.close(fig_acf)

# --- Re-create and save ESS comparison ---
fig_ess, axes_ess = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes_ess[0].bar(
    range(len(ess_tempered)),
    ess_tempered.numpy(),
    width=1.0,
    color="darkorange",
    alpha=0.7,
    label=f"Tempered (N={N_SAMPLES_PT})",
)
if 0 < n_nd < len(ess_tempered):
    axes_ess[0].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes_ess[0].set_ylabel("ESS")
axes_ess[0].set_title("H3: Tempered NUTS (cold chain)", fontsize=12, fontweight="bold")
axes_ess[0].legend(fontsize=10)
axes_ess[0].set_xlim(-0.5, len(ess_tempered) - 0.5)

axes_ess[1].bar(
    range(len(ess_baseline)),
    ess_baseline.numpy(),
    width=1.0,
    color="steelblue",
    alpha=0.7,
    label="H1 baseline (N=10000)",
)
if 0 < n_nd < len(ess_baseline):
    axes_ess[1].axvline(n_nd - 0.5, color="lime", lw=2, ls="--", label="degen boundary")
axes_ess[1].set_xlabel("Eigendirection (sorted by λ, largest first)")
axes_ess[1].set_ylabel("ESS")
axes_ess[1].set_title(
    "H1: Single-chain NUTS (baseline)", fontsize=12, fontweight="bold"
)
axes_ess[1].legend(fontsize=10)
axes_ess[1].set_xlim(-0.5, len(ess_baseline) - 0.5)
fig_ess.suptitle(
    "ESS Comparison: Tempered vs Untempered", fontsize=14, fontweight="bold"
)
fig_ess.tight_layout()
fig_ess.savefig(RESULTS_DIR / "ess_comparison.pdf", bbox_inches="tight", dpi=150)
fig_ess.savefig(RESULTS_DIR / "ess_comparison.png", bbox_inches="tight", dpi=150)
plt.close(fig_ess)

# --- Re-create and save spectral variance plot ---
fig_var, (ax_v1, ax_v2) = plt.subplots(1, 2, figsize=(16, 6))
ax_v1.semilogy(rank_idx.numpy(), var_sorted.numpy(), "o", ms=2, color="teal", alpha=0.6)
ax_v1.axhline(prior_var, color="red", ls="--", lw=1.5)
if 0 < n_nd < len(var_sorted):
    ax_v1.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7)
ax_v1.set_xlabel("Eigendirection rank")
ax_v1.set_ylabel("Variance (log)")
ax_v1.set_title("Sample variance (tempered)")
ax_v1.grid(True, alpha=0.3)

ax_v2.semilogy(
    rank_idx.numpy(), var_sorted.numpy(), "o", ms=2, color="darkorange", alpha=0.6
)
ax_v2.semilogy(
    rank_idx.numpy(), var_h1.numpy(), "o", ms=2, color="steelblue", alpha=0.4
)
ax_v2.axhline(prior_var, color="red", ls="--", lw=1, alpha=0.5)
if 0 < n_nd < len(var_sorted):
    ax_v2.axvline(n_nd, color="lime", lw=2, ls="--", alpha=0.7)
ax_v2.set_xlabel("Eigendirection rank")
ax_v2.set_ylabel("Variance (log)")
ax_v2.set_title("Variance: Tempered vs H1")
ax_v2.grid(True, alpha=0.3)
fig_var.suptitle(
    "Spectral diagnostics: cold-chain exploration", fontsize=13, fontweight="bold"
)
fig_var.tight_layout()
fig_var.savefig(RESULTS_DIR / "spectral_variance.pdf", bbox_inches="tight", dpi=150)
fig_var.savefig(RESULTS_DIR / "spectral_variance.png", bbox_inches="tight", dpi=150)
plt.close(fig_var)

# --- Save numerical results ---
acf50_median = float(acf_at_50.median())
results = {
    "experiment": "04_tempered",
    "hypothesis": "H3: parallel tempering breaks singular mixing barrier",
    "config": {
        "d": int(mle_param.shape[0]),
        "n_chains": N_CHAINS,
        "betas": BETAS,
        "swap_every": SWAP_EVERY,
        "n_warmup": N_WARMUP_PT,
        "n_samples": N_SAMPLES_PT,
        "max_tree_depth": MAX_TREE_DEPTH,
        "sigma_prior": SIGMA_PRIOR,
        "target_accept": TARGET_ACCEPT,
        "dgp_regime": DGP_REGIME,
    },
    "swap_diagnostics": {
        "n_swaps_proposed": result.n_swaps_proposed,
        "n_swaps_accepted": result.n_swaps_accepted,
        "swap_acceptance_rate": float(result.swap_acceptance_rate()),
    },
    "acf": {
        "lag_10_median": float(acf_at_10.median()),
        "lag_50_median": acf50_median,
        "lag_200_median": float(acf_at_200.median()),
        "lag_50_nondegen_median": float(acf50_nd.median()),
        "lag_50_degen_median": float(acf50_dg.median()),
    },
    "ess": {
        "all_min": float(ess_tempered.min()),
        "all_median": float(ess_tempered.median()),
        "all_max": float(ess_tempered.max()),
        "nondegen_min": float(ess_nd_t.min()),
        "nondegen_median": float(ess_nd_t.median()),
        "nondegen_max": float(ess_nd_t.max()),
        "degen_min": float(ess_dg_t.min()),
        "degen_median": float(ess_dg_t.median()),
        "degen_max": float(ess_dg_t.max()),
    },
    "ess_improvement": {
        "median_factor": float(improvement.median()),
        "min_factor": float(improvement.min()),
        "max_factor": float(improvement.max()),
    },
    "elapsed_seconds": elapsed,
    "verdict": "confirmed"
    if acf50_median < 0.3
    else ("inconclusive" if acf50_median < 0.5 else "falsified"),
}

with open(RESULTS_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save raw tensors
torch.save(
    {
        "acf_sorted": acf_sorted,
        "ess_tempered": ess_tempered,
        "ess_baseline": ess_baseline,
        "var_sorted": var_sorted,
        "var_h1": var_h1,
        "eigvals_sorted": eigvals_sorted,
        "n_nd": n_nd,
        "n_degen": n_degen,
        "improvement": improvement,
    },
    RESULTS_DIR / "diagnostics_tensors.pt",
)

print(f"✓ Saved to {RESULTS_DIR}/")
print("  Figures: acf_heatmap_tempered, ess_comparison, spectral_variance (.pdf/.png)")
print("  Data:    results.json, diagnostics_tensors.pt")
print(f"\n{'=' * 70}")
print(f"  H3 VERDICT: {results['verdict'].upper()}")
print(f"{'=' * 70}")
print(
    f"  ACF@50 median = {acf50_median:.3f}  (threshold: < 0.3 confirmed, > 0.5 falsified)"
)
print(f"  ESS median = {float(ess_tempered.median()):.1f} / {N_SAMPLES_PT}")
print(f"  ESS improvement over H1: {float(improvement.median()):.1f}×")
print(f"  Swap acceptance rate: {result.swap_acceptance_rate():.3f}")
print(f"  Wall time: {elapsed / 60:.1f} min")